In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

import anndata as ad
import scanpy as sc
from scipy import sparse

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

import torch
import torch.nn as nn

SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

EMB_DIM_PERT = 128   # pert gene embedding dim (from SVD)
EMB_DIM_OUT = 128   # output gene embedding dim (from SVD)
RANK_R = 32    # low-rank interaction size

DROPOUT = 0.10
LR = 2e-3 * 0.6 * 0.6
WD = 1e-4 * 1.5
EPOCHS = 400
BATCH_GENES = 16 # minibatch over perturbed genes
EVAL_EVERY = 5
PATIENCE = 10 # early stopping patience in eval steps

# Gate parameters should match metric structure
GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

ROOT = Path(".")

# --- added: CV alpha sweep + refit ---
MODEL_SEEDS = [90, 70, 80]
ALPHA_GRID = [0.7, 0.75, 0.778, 0.82, 0.86]
GRAD_CLIP = 1.0
H5AD_PATH = ROOT / "data" / "training_cells.h5ad"  # used ONLY for perts not in gene_columns


In [2]:
means_path = ROOT / "data" / "training_data_means.csv"
valmap_path = ROOT / "data" / "pert_ids_val.csv"
sample_sub_path = ROOT / "data" / "sample_submission.csv"

df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = X_train_means - x_base[None, :] # (80, 5127) delta vs non-targeting

delta_baseline = D_train.mean(axis=0).astype(np.float32)

# pert_id -> gene symbol for leaderboard (first 60)
val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))


Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


In [3]:
AUGMENT_H5AD = True     # turn on/off augmentation
AUG_P = 1.0             # 1.0 = full bootstrap targets, 0.5 = half bootstrap, 0 = off
BOOT_K = 32              # number of bootstrap samples per perturbation
BOOT_M = 256            # cells per bootstrap sample
BOOT_SEED = SEED        # reproducible precompute

# IMPORTANT: to avoid CV leakage, we fit the h5ad->means affine mapping PER FOLD using tr_idx only.
AUG_SLOPE_CLAMP_MIN = 0.2
AUG_SLOPE_CLAMP_MAX = 5.0
AUG_MIN_FIT_PERTS = 8   # if fewer training perts have h5ad cells, we skip augmentation in that fold

EXP_NAME = "baseline_fixed_missing"

# Always fill missing perts from h5ad (DO NOT TURN OFF)
FILL_MISSING_PERTS = True
H5AD_TOPK = 256

# Experiment A: output gene embeddings from h5ad control cells
USE_UOUT_CTRL=True
UOUT_BLEND_ALPHA = 1   # 0 -> pure SVD U_out, 1 -> pure ctrl U_out
UOUT_CTRL_MODE="replace"  # "replace", "concat", or "blend"

# Experiment B: pert embeddings from h5ad control corr for ALL perts (blend with SVD)
USE_ZCTRL_ALL=True
ZCTRL_BETA = 0.5
ZCTRL_TOPK = 512

# Experiment C: GenePT
USE_GENEPT = True
GENEPT_FUSION = "concat"   # "blend" or "concat"
GENEPT_GAMMA = 0.10
GENEPT_WHERE = "z"  # "z", "u", or "both"
GENEPT_DIR = ROOT / "external" / "genept"
GENEPT_WHICH = "gene_protein"  # "gene" or "gene_protein"

# If concat makes training unstable, scale GenePT down
GENEPT_DIM_Z = 64
GENEPT_DIM_U = 64
GENEPT_SCALE_Z = 0.25
GENEPT_SCALE_U = 0.10
GENEPT_MODE = "svd"  # "svd" or "slice"

# DOROTHEA
USE_DOROTHEA = True
DOROTHEA_PATH = ROOT / "external" / "dorothea" / "dorothea.csv"

# SET THIS TO THE FOLDER THAT CONTAINS *protein.info* AND *protein.aliases*
STRING_DIR = ROOT / "external" / "string"

DORO_WHERE = "z"          # "z", "u", or "both"
DORO_FUSION = "concat"       # "concat" or "blend"
DORO_GAMMA = 0.10            # for blend

DORO_DIM_Z = 64
DORO_DIM_U = 64
DORO_SCALE_Z = 0.8
DORO_SCALE_U = 0.10

DORO_INCLUDE_UNSIGNED = True
DORO_L2NORM = True
DORO_MIN_EDGES = 1000
DORO_SEED = SEED

DORO_Z_USE_ROW_THEN_COL = True
DORO_Z_MIN_OUTDEG = 1
DORO_Z_MIN_INDEG = 1
DORO_Z_DEBUG = True
DORO_MODE = "directed_svd"

DORO_SPLIT_POS_NEG = True       # separate stimulation vs inhibition channels
DORO_INCLUDE_UNSIGNED_CH = False  # optional third channel: all directed edges
DORO_SIGNED_ONLY = False        # if True, drop edges with neither stim nor inhib label
DORO_UNSIGNED_WEIGHT = 1.0      # weight for unsigned channel (only if enabled)
_UNIPROT2SYM_CACHE = None

dorothea_z = dorothea_fb_z = dorothea_u = dorothea_fb_u = None

GENEPT_WHERE = "both"
DORO_WHERE   = "both"
USE_STRING_SEQ = True
USE_STRING_SEQ = True
USE_STRING_NET = True
STRING_WHERE   = "both"
STRING_FUSION  = "concat"     # keep concat for now
STRING_DIM_U   = 64
STRING_DIM_Z   = 64           # (see note below)
STRING_SCALE_U = 0.10
STRING_SCALE_Z = 0.25
STRING_MODE    = "svd"
STRING_SPECIES = "9606"
IMPUTE_EXTERNAL = True

In [ ]:
UOUT_CTRL_MODE = "blend"      # do NOT replace
UOUT_BLEND_ALPHA = 0.30       # sweep 0.2, 0.3, 0.4

# Use BOTH, but with safer fusion
GENEPT_WHERE = "both"
DORO_WHERE   = "both"
STRING_WHERE = "both"

# NEW: per-side fusion policy
GENEPT_FUSION_Z = "blend"
GENEPT_FUSION_U = "concat"
DORO_FUSION_Z   = "blend"
DORO_FUSION_U   = "concat"
STRING_FUSION_Z = "blend"
STRING_FUSION_U = "concat"

# Blend strengths for Z (keep small)
GENEPT_GAMMA_Z = 0.10
DORO_GAMMA_Z   = 0.10
STRING_GAMMA_Z = 0.10

# Concat dims for U
GENEPT_DIM_U = 64
DORO_DIM_U   = 64
STRING_DIM_U = 64

# Concat scales for U (small, avoid CV collapse)
GENEPT_SCALE_U = 0.05
DORO_SCALE_U   = 0.10
STRING_SCALE_U = 0.10

# Optional: if GenePT-U coverage is poor, skip it
GENEPT_MIN_U_COVERAGE = 0.85  # 85% coverage threshold

# Impute missing external dict entries (STRING-U mainly)
IMPUTE_EXTERNAL = True

In [ ]:
import pickle
import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler

_H5AD_CACHE = None

def _norm1(v, eps=1e-12):
    return v / (np.linalg.norm(v) + eps)

def _blend_z(z, zext, gamma):
    z = (1.0 - float(gamma)) * z + float(gamma) * zext
    return _norm1(z)

def _maybe_concat_U(U_out, U_block, scale):
    # L2 normalize each row of the block before scaling
    U_block = U_block.astype(np.float32)
    U_block = U_block / (np.linalg.norm(U_block, axis=1, keepdims=True) + 1e-12)
    return np.concatenate([U_out, float(scale) * U_block], axis=1).astype(np.float32)

def load_genept_embeddings(genept_dir="external/genept", which="gene_protein"):
    genept_dir = Path(genept_dir)

    if which == "gene":
        fname = "GenePT_gene_embedding_ada_text.pickle"
    elif which == "gene_protein":
        fname = "GenePT_gene_protein_embedding_model_3_text.pickle"
    else:
        raise ValueError(f"unknown which={which}")

    with open(genept_dir / fname, "rb") as f:
        d = pickle.load(f)

    # Normalize keys and values
    out = {}
    for k, v in d.items():
        kk = str(k).upper()
        out[kk] = np.asarray(v, dtype=np.float32)
    return out

def l2norm_rows(X, eps=1e-12):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)

def reduce_gene_embeddings(g2v, genesU, out_dim, mode="svd", seed=6, l2norm=True):
    """
    g2v: dict {SYMBOL_UPPER: vec}
    genesU: iterable of gene identifiers (already uppercase symbols ideally)
    out_dim: desired reduced dimension
    mode: "svd" or "slice"
    returns: (dict {gene: reduced_vec}, fallback_vec)
    """
    avail = [g for g in genesU if g in g2v]
    if len(avail) < 10:
        print("[genept] too few genes available for reduction:", len(avail))
        return None, None

    X = np.stack([np.asarray(g2v[g], dtype=np.float32) for g in avail], axis=0)

    # Slice mode: cheap, no fit, but usually worse than SVD
    if mode == "slice":
        d = min(int(out_dim), int(X.shape[1]))
        Z = X[:, :d].astype(np.float32)
        if d < out_dim:
            Z = np.concatenate([Z, np.zeros((Z.shape[0], out_dim - d), dtype=np.float32)], axis=1)
        if l2norm:
            Z = l2norm_rows(Z)
        fb = Z.mean(axis=0).astype(np.float32)
        return {avail[i]: Z[i] for i in range(len(avail))}, fb

    # SVD mode: standardize then reduce
    Xs = StandardScaler(with_mean=True, with_std=True).fit_transform(X).astype(np.float32)
    n, d = Xs.shape

    # Clamp components: k must be <= min(n-1, d)
    k = min(int(out_dim), int(d), int(n - 1))
    if k < 1:
        print("[genept] cannot reduce: n, d =", n, d)
        return None, None

    svd = TruncatedSVD(n_components=k, random_state=seed)
    Zk = svd.fit_transform(Xs).astype(np.float32)

    if k < out_dim:
        Z = np.concatenate([Zk, np.zeros((n, out_dim - k), dtype=np.float32)], axis=1)
    else:
        Z = Zk

    if l2norm:
        Z = l2norm_rows(Z)

    fb = Z.mean(axis=0).astype(np.float32)
    return {avail[i]: Z[i] for i in range(len(avail))}, fb

def _pick_pert_col(adata):
    for c in ["sgrna_symbol", "pert_symbol", "pert", "perturbation", "gene", "target_gene"]:
        if c in adata.obs.columns:
            return c
    raise ValueError("Could not find perturbation column in h5ad obs.")

def load_h5ad_ctrl_cache(h5ad_path: Path, gene_columns):
    """
    Cache:
      - Xn (CPM10K + log2(1+x)) sparse
      - ctrl mask
      - var map (GENE->idx)
      - out_idx mapping for the 5127 output genes
      - Xout_z (control cells, standardized per gene)
    """
    global _H5AD_CACHE
    if _H5AD_CACHE is not None:
        return _H5AD_CACHE

    adata = ad.read_h5ad(str(h5ad_path))
    pert_col = _pick_pert_col(adata)

    Xc = adata.X
    if not sp.issparse(Xc):
        Xc = sp.csr_matrix(Xc)
    else:
        Xc = Xc.tocsr()

    cell_sum = np.asarray(Xc.sum(axis=1)).ravel().astype(np.float64)
    scale = (10000.0 / np.clip(cell_sum, 1e-12, None)).astype(np.float64)
    Xn = Xc.multiply(scale[:, None]).tocsr()
    Xn.data = np.log1p(Xn.data) / np.log(2.0)

    obs_pert = adata.obs[pert_col].astype(str).to_numpy()
    obs_pertU = np.char.upper(obs_pert.astype("U"))

    ctrl_mask = (obs_pertU == "NON-TARGETING")
    if int(ctrl_mask.sum()) == 0:
        raise ValueError("No non-targeting control cells found in h5ad.")

    var_names = adata.var_names.astype(str).to_numpy()
    varU = np.char.upper(var_names.astype("U"))
    var = {varU[i]: i for i in range(len(varU))}

    out_idx = np.array([var[str(g).upper()] for g in gene_columns], dtype=np.int64)

    Xn_ctrl = Xn[ctrl_mask]
    Xout = Xn_ctrl[:, out_idx]
    if sp.issparse(Xout):
        Xout = Xout.toarray()
    Xout = Xout.astype(np.float32)

    mu = Xout.mean(axis=0, keepdims=True)
    sd = Xout.std(axis=0, keepdims=True) + 1e-6
    Xout_z = ((Xout - mu) / sd).astype(np.float32)

    _H5AD_CACHE = dict(
        Xn=Xn,
        Xn_ctrl=Xn_ctrl,
        ctrl_mask=ctrl_mask,
        var=var,
        out_idx=out_idx,
        Xout_z=Xout_z,
        obs_pertU=obs_pertU,
    )
    print("[h5ad] cache built:", "n_cells=", Xn.shape[0], "n_genes=", Xn.shape[1], "n_ctrl=", int(ctrl_mask.sum()))
    return _H5AD_CACHE

def build_u_out_from_ctrl(cache, d_out, seed=SEED):
    svd_ctrl = TruncatedSVD(n_components=d_out, random_state=seed)
    svd_ctrl.fit(cache["Xout_z"])
    return svd_ctrl.components_.T.astype(np.float32)  # (G, d_out)

def build_z_from_ctrl_corr(cache, genesU, P_out, topk=256):
    Xout_z = cache["Xout_z"]
    Xn_ctrl = cache["Xn_ctrl"]
    var = cache["var"]

    out = {}
    for gU in genesU:
        j = var.get(gU, None)
        if j is None:
            continue

        xg = Xn_ctrl[:, j]
        if sp.issparse(xg):
            xg = xg.toarray()
        xg = np.asarray(xg).ravel().astype(np.float32)
        xg = (xg - xg.mean()) / (xg.std() + 1e-6)

        corr = (xg[:, None] * Xout_z).mean(axis=0)
        idx = np.argsort(-np.abs(corr))[:topk]
        w = corr[idx].astype(np.float32)

        z = (w[:, None] * P_out[idx]).sum(axis=0)
        z = z / (np.linalg.norm(z) + 1e-12)
        out[gU] = z.astype(np.float32)

    return out

def build_genept_reduced_tables(genept_dict, genes_for_z, genes_for_u, dim_z, dim_u, seed=6,
                               reducer_mode="svd", l2norm=True):
    z_table, z_fb = reduce_gene_embeddings(genept_dict, genes_for_z, dim_z, mode=reducer_mode, seed=seed, l2norm=l2norm)
    u_table, u_fb = reduce_gene_embeddings(genept_dict, genes_for_u, dim_u, mode=reducer_mode, seed=seed, l2norm=l2norm)
    return z_table, z_fb, u_table, u_fb

def concat_genept(base_vec, genept_vec, scale=0.25, l2_after=False):
    if genept_vec is None:
        out = base_vec
    else:
        out = np.concatenate([base_vec, scale * genept_vec], axis=-1)
    if l2_after:
        out = out / (np.linalg.norm(out) + 1e-12)
    return out.astype(np.float32)

def _looks_like_uniprot(x: str) -> bool:
    s = str(x).strip().upper()
    if len(s) < 6 or len(s) > 10:
        return False
    # light heuristic: many UniProt accessions start with O/P/Q or A0A...
    return s[0] in ["O", "P", "Q", "A"]


def _load_string_uniprot_to_symbol_map_from_paths(info_path: Path, ali_path: Path):
    """
    Build UniProt accession -> HGNC symbol using STRING:
      - protein.info: protein_external_id -> preferred_name (symbol)
      - protein.aliases: UniProt accession -> protein_external_id
    """
    # protein.info
    df_info = pd.read_csv(info_path, sep="\t", comment="#", dtype=str)
    cols = list(df_info.columns)
    if cols and cols[0].startswith("#"):
        df_info = df_info.rename(columns={cols[0]: cols[0].lstrip("#")})

    id_col = df_info.columns[0]
    name_col = df_info.columns[1]

    prot = df_info[id_col].astype(str).to_numpy()
    name = df_info[name_col].astype(str).to_numpy()

    # strip taxonomy prefix "9606."
    protU = np.char.upper(np.char.replace(prot.astype("U"), "9606.", ""))
    nameU = np.char.upper(name.astype("U"))
    ensp_to_sym = dict(zip(protU.tolist(), nameU.tolist()))

    # protein.aliases (no header)
    df_ali = pd.read_csv(ali_path, sep="\t", comment="#", dtype=str, header=None)
    if df_ali.shape[1] < 3:
        raise ValueError("[dorothea] STRING aliases file has unexpected format (need >=3 cols).")

    prot_id = df_ali.iloc[:, 0].astype(str).to_numpy()
    alias   = df_ali.iloc[:, 1].astype(str).to_numpy()
    src     = df_ali.iloc[:, 2].astype(str).to_numpy()

    prot_idU = np.char.upper(np.char.replace(prot_id.astype("U"), "9606.", ""))
    aliasU   = np.char.upper(alias.astype("U"))
    srcU     = np.char.upper(src.astype("U"))

    keep = np.char.find(srcU, "UNIPROT") >= 0
    if int(keep.sum()) == 0:
        raise ValueError("[dorothea] No UniProt aliases found in STRING aliases file.")

    m = {}
    for a, p in zip(aliasU[keep], prot_idU[keep]):
        sym = ensp_to_sym.get(p, None)
        if sym is None:
            continue
        if a not in m:
            m[a] = sym
    return m


def _get_uniprot_to_symbol_map():
    """
    Cached UniProt->symbol map.
    Uses explicit STRING file paths:
      external/string/9606.protein.info.v12.0.txt
      external/string/9606.protein.aliases.v12.0.txt
    """
    global _UNIPROT2SYM_CACHE
    if _UNIPROT2SYM_CACHE is not None:
        return _UNIPROT2SYM_CACHE

    string_dir = ROOT / "external" / "string"
    info_path = string_dir / "9606.protein.info.v12.0.txt"
    ali_path  = string_dir / "9606.protein.aliases.v12.0.txt"

    if not info_path.exists():
        raise ValueError(f"[dorothea] Missing STRING info file: {info_path}")
    if not ali_path.exists():
        raise ValueError(f"[dorothea] Missing STRING aliases file: {ali_path}")

    m = _load_string_uniprot_to_symbol_map_from_paths(info_path, ali_path)
    print("[dorothea] built UniProt->symbol map from STRING:", len(m))
    _UNIPROT2SYM_CACHE = m
    return _UNIPROT2SYM_CACHE


def _read_dorothea_df(path: Path) -> pd.DataFrame:
    # auto-detect delimiter; normalize header names; strip BOM
    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [str(c).replace("\ufeff", "").strip().lower() for c in df.columns]
    return df


def load_dorothea_edges_signed_unsigned(dorothea_csv: Path, genesU_set: set):
    """
    Returns edges inside genesU_set:
      srcK, tgtK: uppercase symbols (mapped from UniProt via STRING if needed)
      stimK: bool
      inhibK: bool
      undK: bool (directed)
    """
    df = _read_dorothea_df(dorothea_csv)

    if ("source" not in df.columns) or ("target" not in df.columns):
        raise ValueError(f"[dorothea] could not find source/target in columns: {list(df.columns)[:30]}")

    src = df["source"].astype(str).str.upper().to_numpy()
    tgt = df["target"].astype(str).str.upper().to_numpy()

    # detect UniProt-like IDs and map via STRING if needed
    sample = src[:50].tolist() + tgt[:50].tolist()
    uniprotish = np.mean([_looks_like_uniprot(x) and (x not in genesU_set) for x in sample]) > 0.5
    if uniprotish:
        m = _get_uniprot_to_symbol_map()
        if len(m) > 0:
            src = np.array([m.get(s, s) for s in src], dtype="U")
            tgt = np.array([m.get(t, t) for t in tgt], dtype="U")

    stim_cols  = [c for c in ["consensus_stimulation", "is_stimulation"] if c in df.columns]
    inhib_cols = [c for c in ["consensus_inhibition", "is_inhibition"] if c in df.columns]

    stim = np.zeros(len(df), dtype=bool)
    inhib = np.zeros(len(df), dtype=bool)
    for c in stim_cols:
        stim |= df[c].astype(bool).to_numpy()
    for c in inhib_cols:
        inhib |= df[c].astype(bool).to_numpy()

    if "is_directed" in df.columns:
        und = df["is_directed"].astype(bool).to_numpy()
    else:
        und = np.ones(len(df), dtype=bool)

    keep = np.array([(s in genesU_set) and (t in genesU_set) for s, t in zip(src, tgt)], dtype=bool)

    print("[dorothea] parsed cols:", list(df.columns)[:12])
    print("[dorothea] edges loaded:", len(df), "kept(in-gene-set):", int(keep.sum()))
    if uniprotish:
        print("[dorothea] detected UniProt-like IDs:", True, "| mapping size:", len(_UNIPROT2SYM_CACHE or {}))

    return src[keep], tgt[keep], stim[keep], inhib[keep], und[keep]


def _svd_rowcol_embeddings(A: sp.csr_matrix, k: int, seed: int):
    """
    For sparse A (G x G):
      row_emb = UΣ  via svd.fit_transform(A)
      col_emb = VΣ  via components_.T * singular_values
    Returns (row_emb, col_emb) each (G, k), padded as needed.
    """
    G = A.shape[0]
    k = int(k)
    k_eff = int(min(max(1, k), G - 1))

    if A.nnz < 10 or k_eff < 1:
        return np.zeros((G, k), dtype=np.float32), np.zeros((G, k), dtype=np.float32)

    svd = TruncatedSVD(n_components=k_eff, random_state=seed)
    row_eff = svd.fit_transform(A).astype(np.float32)  # (G, k_eff) = UΣ
    col_eff = (svd.components_.T.astype(np.float32) *
               svd.singular_values_.astype(np.float32)[None, :]).astype(np.float32)  # (G, k_eff) = VΣ

    if k_eff < k:
        pad = np.zeros((G, k - k_eff), dtype=np.float32)
        row = np.concatenate([row_eff, pad], axis=1)
        col = np.concatenate([col_eff, pad], axis=1)
    else:
        row = row_eff[:, :k]
        col = col_eff[:, :k]

    return row, col


def build_dorothea_directed_svd_embeddings(
    dorothea_csv: Path,
    gene_columns,
    genes_needed,
    dim_z: int,
    dim_u: int,
    seed: int = 6,
    split_pos_neg: bool = True,
    include_unsigned_ch: bool = False,
    signed_only: bool = False,
    unsigned_weight: float = 1.0,
    l2norm: bool = True,
    min_edges: int = 1000,
):
    """
    Directed TF->target SVD.

    Z-space:
      - Z_row: row embeddings (source/regulator) = UΣ
      - Z_col: col embeddings (target)          = VΣ
      - Z per pert: row if outdeg>0 else col if indeg>0 else fallback (fb_u)

    U_out-space:
      - Ug: col embeddings in dim_u (targets) for optional U block
    """
    genesU = [str(g).upper() for g in gene_columns]
    genesU_set = set(genesU)
    g2i = {g: i for i, g in enumerate(genesU)}
    G = len(genesU)

    src, tgt, stim, inhib, und = load_dorothea_edges_signed_unsigned(dorothea_csv, genesU_set=genesU_set)
    if len(src) < int(min_edges):
        print("[dorothea] too few usable edges, skipping:", len(src))
        return None, None, None, None

    if signed_only:
        keep_s = stim | inhib
        src = src[keep_s]; tgt = tgt[keep_s]
        stim = stim[keep_s]; inhib = inhib[keep_s]; und = und[keep_s]
        print("[dorothea] signed_only kept edges:", len(src))

    keep_d = und.astype(bool)
    src = src[keep_d]; tgt = tgt[keep_d]
    stim = stim[keep_d]; inhib = inhib[keep_d]

    if len(src) < int(min_edges):
        print("[dorothea] too few directed edges after filtering, skipping:", len(src))
        return None, None, None, None

    ii = np.array([g2i[s] for s in src], dtype=np.int64)
    jj = np.array([g2i[t] for t in tgt], dtype=np.int64)

    outdeg = np.bincount(ii, minlength=G).astype(np.int32)
    indeg  = np.bincount(jj, minlength=G).astype(np.int32)

    def mkA(mask, val=1.0):
        m = mask.astype(bool)
        if int(m.sum()) == 0:
            return sp.csr_matrix((G, G), dtype=np.float32)
        data = np.full(int(m.sum()), float(val), dtype=np.float32)
        return sp.csr_matrix((data, (ii[m], jj[m])), shape=(G, G), dtype=np.float32)

    dim_z = int(dim_z)
    dim_u = int(dim_u)

    # Allocate dims for Z
    if split_pos_neg:
        if include_unsigned_ch:
            k_any_z = max(1, dim_z // 3)
            rem_z = dim_z - k_any_z
            k_pos_z = max(1, rem_z // 2)
            k_neg_z = max(1, rem_z - k_pos_z)
        else:
            k_pos_z = max(1, dim_z // 2)
            k_neg_z = max(1, dim_z - k_pos_z)
            k_any_z = 0
    else:
        k_pos_z = dim_z; k_neg_z = 0; k_any_z = 0

    # Allocate dims for U
    if split_pos_neg:
        if include_unsigned_ch:
            k_any_u = max(1, dim_u // 3)
            rem_u = dim_u - k_any_u
            k_pos_u = max(1, rem_u // 2)
            k_neg_u = max(1, rem_u - k_pos_u)
        else:
            k_pos_u = max(1, dim_u // 2)
            k_neg_u = max(1, dim_u - k_pos_u)
            k_any_u = 0
    else:
        k_pos_u = dim_u; k_neg_u = 0; k_any_u = 0

    if split_pos_neg:
        A_pos = mkA(stim, val=1.0)
        A_neg = mkA(inhib, val=1.0)
        A_any = mkA(np.ones_like(stim, dtype=bool), val=float(unsigned_weight)) if include_unsigned_ch else None

        Zr_pos, Zc_pos = _svd_rowcol_embeddings(A_pos, k_pos_z, seed=seed)
        Zr_neg, Zc_neg = _svd_rowcol_embeddings(A_neg, k_neg_z, seed=seed + 1)

        Z_row = np.concatenate([Zr_pos, Zr_neg], axis=1).astype(np.float32)
        Z_col = np.concatenate([Zc_pos, Zc_neg], axis=1).astype(np.float32)

        if include_unsigned_ch and A_any is not None:
            Zr_any, Zc_any = _svd_rowcol_embeddings(A_any, k_any_z, seed=seed + 2)
            Z_row = np.concatenate([Z_row, Zr_any], axis=1).astype(np.float32)
            Z_col = np.concatenate([Z_col, Zc_any], axis=1).astype(np.float32)

        Z_row = Z_row[:, :dim_z] if Z_row.shape[1] >= dim_z else np.concatenate([Z_row, np.zeros((G, dim_z - Z_row.shape[1]), np.float32)], axis=1)
        Z_col = Z_col[:, :dim_z] if Z_col.shape[1] >= dim_z else np.concatenate([Z_col, np.zeros((G, dim_z - Z_col.shape[1]), np.float32)], axis=1)

        # U space uses COL embeddings
        _, Uc_pos = _svd_rowcol_embeddings(A_pos, k_pos_u, seed=seed + 10)
        _, Uc_neg = _svd_rowcol_embeddings(A_neg, k_neg_u, seed=seed + 11)
        Ug = np.concatenate([Uc_pos, Uc_neg], axis=1).astype(np.float32)

        if include_unsigned_ch and A_any is not None:
            _, Uc_any = _svd_rowcol_embeddings(A_any, k_any_u, seed=seed + 12)
            Ug = np.concatenate([Ug, Uc_any], axis=1).astype(np.float32)

        Ug = Ug[:, :dim_u] if Ug.shape[1] >= dim_u else np.concatenate([Ug, np.zeros((G, dim_u - Ug.shape[1]), np.float32)], axis=1)

    else:
        ws = np.zeros(len(src), dtype=np.float32)
        ws[stim] = 1.0
        ws[inhib] = -1.0
        A = sp.csr_matrix((ws, (ii, jj)), shape=(G, G), dtype=np.float32)
        Z_row, Z_col = _svd_rowcol_embeddings(A, dim_z, seed=seed)
        Ug = Z_col[:, :dim_u] if dim_u <= Z_col.shape[1] else np.concatenate([Z_col, np.zeros((G, dim_u - Z_col.shape[1]), np.float32)], axis=1)

    if l2norm:
        Z_row = l2norm_rows(Z_row)
        Z_col = l2norm_rows(Z_col)
        Ug = l2norm_rows(Ug)

    dorothea_u = {genesU[i]: Ug[i].copy() for i in range(G)}
    fb_u = Ug.mean(axis=0).astype(np.float32)
    
    # DORO_Z_USE_ROW_THEN_COL, DORO_Z_MIN_OUTDEG, DORO_Z_MIN_INDEG, DORO_Z_DEBUG
    dorothea_u = {genesU[i]: Ug[i].copy() for i in range(G)}
    fb_u = Ug.mean(axis=0).astype(np.float32)

    # Z fallback must match dim_z, not dim_u
    fb_z0 = (0.5 * (Z_row.mean(axis=0) + Z_col.mean(axis=0))).astype(np.float32)

    dorothea_z = {}
    used_row = used_col = used_fb = 0

    for p in genes_needed:
        pU = str(p).upper()
        if pU not in g2i:
            dorothea_z[pU] = fb_z0.copy()
            used_fb += 1
            continue

        idx = g2i[pU]
        if DORO_Z_USE_ROW_THEN_COL and (outdeg[idx] >= int(DORO_Z_MIN_OUTDEG)):
            dorothea_z[pU] = Z_row[idx].copy()
            used_row += 1
        elif DORO_Z_USE_ROW_THEN_COL and (indeg[idx] >= int(DORO_Z_MIN_INDEG)):
            dorothea_z[pU] = Z_col[idx].copy()
            used_col += 1
        else:
            dorothea_z[pU] = fb_z0.copy()
            used_fb += 1

    fb_z = np.stack(list(dorothea_z.values()), axis=0).mean(axis=0).astype(np.float32)

    if DORO_Z_DEBUG:
        trainU = [str(g).upper() for g in train_genes.tolist()] if "train_genes" in globals() else []
        if trainU:
            tr_row = sum((g in g2i) and (outdeg[g2i[g]] >= int(DORO_Z_MIN_OUTDEG)) for g in trainU)
            tr_col = sum((g in g2i) and (outdeg[g2i[g]] < int(DORO_Z_MIN_OUTDEG)) and (indeg[g2i[g]] >= int(DORO_Z_MIN_INDEG)) for g in trainU)
            tr_fb  = len(trainU) - tr_row - tr_col
            print(f"[dorothea] Z policy (train perts): row={tr_row} col={tr_col} fb={tr_fb} (n={len(trainU)})")

        print(f"[dorothea] Z policy (genes_needed): row={used_row} col={used_col} fb={used_fb} (n={len(genes_needed)})")
        print("[dorothea] degree stats:",
              "outdeg>0:", int((outdeg > 0).sum()),
              "indeg>0:", int((indeg > 0).sum()),
              "G:", G)

    print("[dorothea] directed_svd built:",
          "dim_z=", dim_z, "dim_u=", dim_u,
          "split_pos_neg=", split_pos_neg,
          "unsigned_ch=", include_unsigned_ch,
          "signed_only=", signed_only)
    return dorothea_z, fb_z, dorothea_u, fb_u

# =============================
# STRING helpers (required)
# Provides:
#  - get_string_id_to_symbol_map
#  - load_string_raw_embeddings
#  - build_string_reduced_tables
# =============================

from pathlib import Path
import numpy as np
import pandas as pd

_STRING_ID2SYM_CACHE = None
_STRING_RAW_CACHE = {}  # key=(species, kind) -> (gene2vec_raw, fb_raw)

def _find_first_glob(dirpath: Path, patterns):
    for pat in patterns:
        hits = sorted(dirpath.glob(pat))
        if hits:
            return hits[0]
    return None

def get_string_id_to_symbol_map(string_dir: Path, species: str = "9606"):
    """
    Loads mapping: STRING protein_id -> preferred gene symbol (uppercased)
    Uses: 9606.protein.info*.txt(.gz)
    """
    global _STRING_ID2SYM_CACHE
    if _STRING_ID2SYM_CACHE is not None:
        return _STRING_ID2SYM_CACHE

    info_path = _find_first_glob(
        Path(string_dir),
        patterns=[
            f"{species}.protein.info*.txt.gz",
            f"{species}.protein.info*.txt",
        ],
    )
    if info_path is None:
        raise FileNotFoundError(f"STRING protein.info not found under {string_dir}")

    df = pd.read_csv(info_path, sep="\t", comment="#", dtype=str)
    cols = list(df.columns)
    if cols and cols[0].startswith("#"):
        df = df.rename(columns={cols[0]: cols[0].lstrip("#")})

    id_col = df.columns[0]
    name_col = df.columns[1]

    prot_ids = df[id_col].astype(str).to_numpy()
    names = df[name_col].astype(str).str.upper().to_numpy()

    _STRING_ID2SYM_CACHE = dict(zip(prot_ids, names))
    return _STRING_ID2SYM_CACHE

def load_string_raw_embeddings(string_dir: Path, kind: str, species: str = "9606"):
    """
    kind: 'sequence' or 'network'
    Expects H5 datasets:
      - embeddings: (N, d)
      - proteins:   (N,)
    Returns:
      gene2vec: dict[SYMBOL_U] -> (d,) float32
      fb: (d,) float32 mean embedding over proteins
    """
    key = (species, kind)
    if key in _STRING_RAW_CACHE:
        return _STRING_RAW_CACHE[key]

    import h5py

    if kind not in ["sequence", "network"]:
        raise ValueError("kind must be 'sequence' or 'network'")

    h5_path = _find_first_glob(
        Path(string_dir),
        patterns=[
            f"{species}.protein.{kind}.embeddings*.h5",
            f"{species}.protein.{kind}.embeddings*.hdf5",
        ],
    )
    if h5_path is None:
        raise FileNotFoundError(f"STRING {kind} embeddings h5 not found under {string_dir}")

    id2sym = get_string_id_to_symbol_map(Path(string_dir), species=species)

    with h5py.File(h5_path, "r") as f:
        E = f["embeddings"][:].astype(np.float32)
        P = f["proteins"][:]

    # proteins may be bytes
    prots = []
    for p in P:
        if isinstance(p, (bytes, np.bytes_)):
            prots.append(p.decode("utf-8"))
        else:
            prots.append(str(p))

    # Map protein_id -> gene symbol, average duplicates
    sums = {}
    cnts = {}
    for pid, vec in zip(prots, E):
        sym = id2sym.get(pid, None)
        if sym is None:
            continue
        gU = str(sym).upper()
        if gU not in sums:
            sums[gU] = vec.copy()
            cnts[gU] = 1
        else:
            sums[gU] += vec
            cnts[gU] += 1

    gene2vec = {gU: (sums[gU] / float(cnts[gU])).astype(np.float32) for gU in sums.keys()}
    fb = (E.mean(axis=0).astype(np.float32) if E.shape[0] > 0 else None)

    _STRING_RAW_CACHE[key] = (gene2vec, fb)
    return gene2vec, fb

def build_string_reduced_tables(string_dir: Path, genes_list, out_dim: int, kind: str,
                                mode: str = "svd", seed: int = 0, species: str = "9606"):
    """
    Returns: (red_dict, fallback_vec)
    """
    raw, fb_raw = load_string_raw_embeddings(Path(string_dir), kind=kind, species=species)
    red, fb = reduce_gene_embeddings(raw, [str(g).upper() for g in genes_list],
                                     out_dim=int(out_dim), mode=mode, seed=int(seed))
    return red, fb

# =========================
# Alignment helpers
# =========================

def _l2row(X, eps=1e-12):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)

def mean_row_cos(A, B):
    A = _l2row(A.astype(np.float32))
    B = _l2row(B.astype(np.float32))
    return float(np.mean(np.sum(A * B, axis=1)))

def orth_procrustes(A, B):
    """
    Find R (dxd) orthogonal that aligns A -> B in least squares:
      argmin_R ||A R - B||, s.t. R^T R = I
    A,B: (n,d)
    """
    A = A.astype(np.float32)
    B = B.astype(np.float32)
    M = A.T @ B
    U, _, Vt = np.linalg.svd(M, full_matrices=False)
    R = (U @ Vt).astype(np.float32)
    return R

def fit_ridge_map(X, Y, lam=1e-2):
    """
    Fit W: (d_in, d_out) minimizing ||X W - Y||^2 + lam||W||^2
    """
    X = X.astype(np.float32)
    Y = Y.astype(np.float32)
    d_in = X.shape[1]
    A = X.T @ X + float(lam) * np.eye(d_in, dtype=np.float32)
    B = X.T @ Y
    W = np.linalg.solve(A, B).astype(np.float32)
    return W

def build_align_map_from_dicts(ext_dict, base_dict, keysU, lam=1e-2, min_pairs=128, name="src"):
    """
    ext_dict: dict[str]->(d_in,)
    base_dict: dict[str]->(d_out,)
    keysU: list[str] candidate keys
    Returns W or None
    """
    pairs = [k for k in keysU if (k in ext_dict) and (k in base_dict)]
    if len(pairs) < min_pairs:
        print(f"[align] {name}: too few pairs ({len(pairs)}) skip")
        return None

    X = np.vstack([ext_dict[k] for k in pairs]).astype(np.float32)
    Y = np.vstack([base_dict[k] for k in pairs]).astype(np.float32)
    W = fit_ridge_map(X, Y, lam=lam)

    # quick sanity: how well does it reconstruct base?
    Yhat = X @ W
    cos = mean_row_cos(Yhat, Y)
    print(f"[align] {name}: pairs={len(pairs)}  recon_cos={cos:.4f}")
    return W

Build gene embeddings from D_train (SVD on signed-log transformed deltas). Build embeddings for output genes (gene_columns) via SVD on (80, 5127). Pert embeddings are looked up by gene symbol in gene_columns, otherwise fallback to the mean embedding.

In [ ]:
# ============================================================
# EMBEDDINGS BUILD CELL (stable BOTH)
# Z: blend (keeps d_pert)
# U: concat (can be wide)
# ============================================================

import numpy as np

val_targets = df_valmap["pert"].astype(str).tolist()

genes_needed = sorted(set([str(g).upper() for g in train_genes.tolist()] +
                          [str(g).upper() for g in val_targets]))

geneU = pd.Index([str(g).upper() for g in gene_columns])

X = D_train.astype(np.float32, copy=True)
X = np.sign(X) * np.log2(1.0 + np.abs(X))

svd = TruncatedSVD(n_components=max(EMB_DIM_PERT, EMB_DIM_OUT), random_state=SEED)
svd.fit(X)

gene_emb_all = svd.components_.T.astype(np.float32)  # (G, k)
k_svd = gene_emb_all.shape[1]
d_pert = min(int(EMB_DIM_PERT), int(k_svd))
d_out  = min(int(EMB_DIM_OUT),  int(k_svd))

print("SVD k:", k_svd, "gene_emb_all:", gene_emb_all.shape)
print("Effective dims:", "d_pert=", d_pert, "d_out=", d_out)

gene2emb_pert_svd = {gene_columns[i].upper(): gene_emb_all[i, :d_pert].copy()
                     for i in range(len(gene_columns))}
gene2emb_out_svd  = {gene_columns[i].upper(): gene_emb_all[i, :d_out ].copy()
                     for i in range(len(gene_columns))}

emb_fallback_pert = gene_emb_all[:, :d_pert].mean(axis=0).astype(np.float32)
emb_fallback_out  = gene_emb_all[:, :d_out ].mean(axis=0).astype(np.float32)

# -----------------------------
# Missing perts
# -----------------------------
missing_train = [g for g in train_genes.tolist() if str(g).upper() not in geneU]
missing_val   = [g for g in val_targets           if str(g).upper() not in geneU]
missing_allU  = sorted(set([str(g).upper() for g in (missing_train + missing_val)]))

if missing_train:
    print(f"train perts not in gene_columns: {len(missing_train)}. Example: {missing_train[:12]}")
if missing_val:
    print(f"val perts not in gene_columns: {len(missing_val)}. Example: {missing_val[:12]}")

# -----------------------------
# h5ad: missing pert fill + zctrl_all + U_ctrl
# -----------------------------
missing_pert_h5ad = {}
zctrl_all = {}
U_ctrl = None

if (FILL_MISSING_PERTS and len(missing_allU) > 0) or USE_ZCTRL_ALL or USE_UOUT_CTRL:
    cache = load_h5ad_ctrl_cache(H5AD_PATH, gene_columns)
    P_out = gene_emb_all[:, :d_pert].astype(np.float32)

    if FILL_MISSING_PERTS and len(missing_allU) > 0:
        tmp = build_z_from_ctrl_corr(cache, missing_allU, P_out, topk=H5AD_TOPK)
        missing_pert_h5ad.update(tmp)
        print("[h5ad] embedded missing perts:", len(missing_pert_h5ad), "of", len(missing_allU))

    if USE_ZCTRL_ALL:
        tmp = build_z_from_ctrl_corr(cache, genes_needed, P_out, topk=ZCTRL_TOPK)
        zctrl_all.update(tmp)
        print("[h5ad] embedded ALL perts for blending:", len(zctrl_all), "of", len(genes_needed))

    if USE_UOUT_CTRL:
        U_ctrl = build_u_out_from_ctrl(cache, d_out=d_out, seed=SEED)
        print("[h5ad] U_ctrl:", U_ctrl.shape)

GENEPT_FUSION_Z = globals().get("GENEPT_FUSION_Z", globals().get("GENEPT_FUSION", "concat"))
GENEPT_FUSION_U = globals().get("GENEPT_FUSION_U", globals().get("GENEPT_FUSION", "concat"))
DORO_FUSION_Z   = globals().get("DORO_FUSION_Z",   globals().get("DORO_FUSION",   "concat"))
DORO_FUSION_U   = globals().get("DORO_FUSION_U",   globals().get("DORO_FUSION",   "concat"))
STRING_FUSION_Z = globals().get("STRING_FUSION_Z", globals().get("STRING_FUSION", "concat"))
STRING_FUSION_U = globals().get("STRING_FUSION_U", globals().get("STRING_FUSION", "concat"))

GENEPT_GAMMA_Z  = float(globals().get("GENEPT_GAMMA_Z", globals().get("GENEPT_GAMMA", 0.10)))
DORO_GAMMA_Z    = float(globals().get("DORO_GAMMA_Z",   globals().get("DORO_GAMMA",   0.10)))
STRING_GAMMA_Z  = float(globals().get("STRING_GAMMA_Z", globals().get("STRING_GAMMA", 0.10)))

GENEPT_MIN_U_COVERAGE = float(globals().get("GENEPT_MIN_U_COVERAGE", 0.0))

# -----------------------------
# Helpers
# -----------------------------
def _norm1(v, eps=1e-12):
    return v / (np.linalg.norm(v) + eps)

def _blend_z(z, zext, gamma):
    return _norm1((1.0 - float(gamma)) * z + float(gamma) * zext)

def _l2row(X, eps=1e-12):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)

def _concat_u(U_out, U_block, scale):
    U_block = _l2row(U_block.astype(np.float32))
    return np.concatenate([U_out, float(scale) * U_block], axis=1).astype(np.float32)

# -----------------------------
# GenePT: build Z/U tables with correct per-side dims
# -----------------------------
genept_red_z = genept_fb_z = genept_red_u = genept_fb_u = None

if USE_GENEPT:
    g2v = load_genept_embeddings(GENEPT_DIR, which=GENEPT_WHICH)

    dim_z = int(d_pert) if (GENEPT_FUSION_Z == "blend") else int(GENEPT_DIM_Z)
    dim_u = int(d_out)  if (GENEPT_FUSION_U == "blend") else int(GENEPT_DIM_U)

    if (GENEPT_WHERE in ["z", "both"]):
        genept_red_z, genept_fb_z = reduce_gene_embeddings(
            g2v, genes_needed, out_dim=dim_z, mode=GENEPT_MODE, seed=SEED
        )
    if (GENEPT_WHERE in ["u", "both"]):
        genept_red_u, genept_fb_u = reduce_gene_embeddings(
            g2v, [str(g).upper() for g in gene_columns], out_dim=dim_u, mode=GENEPT_MODE, seed=SEED
        )

    print("[genept] reduced:",
          "z_avail=", (0 if genept_red_z is None else len(genept_red_z)),
          "u_avail=", (0 if genept_red_u is None else len(genept_red_u)),
          "dim_z=", (None if genept_fb_z is None else genept_fb_z.shape[0]),
          "dim_u=", (None if genept_fb_u is None else genept_fb_u.shape[0]))

# -----------------------------
# DoRothEA: build Z/U tables with correct per-side dims
# -----------------------------
dorothea_z = dorothea_fb_z = dorothea_u = dorothea_fb_u = None

if USE_DOROTHEA:
    dim_z = int(d_pert) if (DORO_FUSION_Z == "blend") else int(DORO_DIM_Z)
    dim_u = int(d_out)  if (DORO_FUSION_U == "blend") else int(DORO_DIM_U)

    if DORO_MODE == "directed_svd":
        dorothea_z, dorothea_fb_z, dorothea_u, dorothea_fb_u = build_dorothea_directed_svd_embeddings(
            DOROTHEA_PATH,
            gene_columns=gene_columns,
            genes_needed=genes_needed,
            dim_z=dim_z,
            dim_u=dim_u,
            seed=DORO_SEED,
            split_pos_neg=bool(DORO_SPLIT_POS_NEG),
            include_unsigned_ch=bool(DORO_INCLUDE_UNSIGNED_CH),
            signed_only=bool(DORO_SIGNED_ONLY),
            unsigned_weight=float(DORO_UNSIGNED_WEIGHT),
            l2norm=bool(DORO_L2NORM),
            min_edges=int(DORO_MIN_EDGES),
        )
    else:
        raise ValueError("DORO_MODE must be 'directed_svd' here")

    print("[dorothea] tables:",
          "z_avail=", (0 if dorothea_z is None else len(dorothea_z)),
          "u_avail=", (0 if dorothea_u is None else len(dorothea_u)),
          "dim_z=", (None if dorothea_fb_z is None else dorothea_fb_z.shape[0]),
          "dim_u=", (None if dorothea_fb_u is None else dorothea_fb_u.shape[0]))

# -----------------------------
# STRING: build Z/U tables with correct per-side dims
# -----------------------------
string_seq_z = string_seq_fb_z = string_seq_u = string_seq_fb_u = None
string_net_z = string_net_fb_z = string_net_u = string_net_fb_u = None

if (globals().get("USE_STRING_SEQ", False) or globals().get("USE_STRING_NET", False)):
    dim_z = int(d_pert) if (STRING_FUSION_Z == "blend") else int(STRING_DIM_Z)
    dim_u = int(d_out)  if (STRING_FUSION_U == "blend") else int(STRING_DIM_U)

    if USE_STRING_SEQ:
        if STRING_WHERE in ["z", "both"]:
            string_seq_z, string_seq_fb_z = build_string_reduced_tables(
                STRING_DIR, genes_needed, out_dim=dim_z, kind="sequence",
                mode=STRING_MODE, seed=SEED, species=STRING_SPECIES
            )
        if STRING_WHERE in ["u", "both"]:
            string_seq_u, string_seq_fb_u = build_string_reduced_tables(
                STRING_DIR, [str(g).upper() for g in gene_columns], out_dim=dim_u, kind="sequence",
                mode=STRING_MODE, seed=SEED, species=STRING_SPECIES
            )

    if USE_STRING_NET:
        if STRING_WHERE in ["z", "both"]:
            string_net_z, string_net_fb_z = build_string_reduced_tables(
                STRING_DIR, genes_needed, out_dim=dim_z, kind="network",
                mode=STRING_MODE, seed=SEED, species=STRING_SPECIES
            )
        if STRING_WHERE in ["u", "both"]:
            string_net_u, string_net_fb_u = build_string_reduced_tables(
                STRING_DIR, [str(g).upper() for g in gene_columns], out_dim=dim_u, kind="network",
                mode=STRING_MODE, seed=SEED, species=STRING_SPECIES
            )

    print("[string] reduced:",
          "seq_z=", (0 if string_seq_z is None else len(string_seq_z)),
          "seq_u=", (0 if string_seq_u is None else len(string_seq_u)),
          "net_z=", (0 if string_net_z is None else len(string_net_z)),
          "net_u=", (0 if string_net_u is None else len(string_net_u)),
          "dim_z=", (None if string_seq_fb_z is None else string_seq_fb_z.shape[0]),
          "dim_u=", (None if string_seq_fb_u is None else string_seq_fb_u.shape[0]))

# -----------------------------
# Optional: kNN impute missing external dicts so concat blocks are complete
# (STRING-U is the main gap)
# -----------------------------
def _impute_knn(ext_dict, keysU, base_mat, k=64, tau=0.07):
    if ext_dict is None:
        return 0
    have = [i for i,g in enumerate(keysU) if g in ext_dict]
    miss = [i for i,g in enumerate(keysU) if g not in ext_dict]
    if len(miss) == 0 or len(have) < 8:
        return 0

    B = base_mat.astype(np.float32)
    B = _l2row(B)

    Bh = B[np.array(have)]
    # ext dim
    d_ext = None
    for g in ext_dict.keys():
        d_ext = int(np.asarray(ext_dict[g]).shape[0])
        break
    Eh = np.vstack([ext_dict[keysU[i]] for i in have]).astype(np.float32)

    filled = 0
    for i in miss:
        sims = (Bh @ B[i]).astype(np.float32)
        kk = min(int(k), sims.shape[0])
        top = np.argpartition(-sims, kk - 1)[:kk]
        s = sims[top]
        s = s - np.max(s)
        w = np.exp(s / float(tau)).astype(np.float32)
        w = w / (w.sum() + 1e-12)
        ext_dict[keysU[i]] = (w[:, None] * Eh[top]).sum(axis=0).astype(np.float32)
        filled += 1
    return filled

IMPUTE_EXTERNAL = bool(globals().get("IMPUTE_EXTERNAL", False))

if IMPUTE_EXTERNAL:
    gene_columnsU = [str(g).upper() for g in gene_columns]
    genes_neededU = [str(g).upper() for g in genes_needed]

    # base matrices for kNN space
    baseU = np.vstack([gene2emb_out_svd.get(g, emb_fallback_out) for g in gene_columnsU]).astype(np.float32)
    baseZ = np.vstack([
        (gene2emb_pert_svd[g] if g in gene2emb_pert_svd else (missing_pert_h5ad.get(g, emb_fallback_pert)))
        for g in genes_neededU
    ]).astype(np.float32)

    # Impute U dicts
    for nm, dct in [("string_seq_u", string_seq_u), ("string_net_u", string_net_u), ("genept_red_u", genept_red_u)]:
        if dct is None:
            continue
        before = sum([1 for g in gene_columnsU if g in dct])
        filled = _impute_knn(dct, gene_columnsU, baseU)
        after  = sum([1 for g in gene_columnsU if g in dct])
        if filled > 0:
            print(f"[impute U] {nm}: before={before}/{len(gene_columnsU)} filled={filled} after={after}/{len(gene_columnsU)}")

    # Impute Z dicts
    for nm, dct in [("genept_red_z", genept_red_z), ("string_seq_z", string_seq_z), ("string_net_z", string_net_z)]:
        if dct is None:
            continue
        before = sum([1 for g in genes_neededU if g in dct])
        filled = _impute_knn(dct, genes_neededU, baseZ)
        after  = sum([1 for g in genes_neededU if g in dct])
        if filled > 0:
            print(f"[impute Z] {nm}: before={before}/{len(genes_neededU)} filled={filled} after={after}/{len(genes_neededU)}")

# -----------------------------
# emb_pert: Z is BLENDED, so Z_train stays (80, d_pert)
# -----------------------------
def emb_pert(g: str) -> np.ndarray:
    gU = str(g).upper()

    # Base
    if gU in gene2emb_pert_svd:
        z = gene2emb_pert_svd[gU]
    elif gU in missing_pert_h5ad:
        z = missing_pert_h5ad[gU]
    else:
        z = emb_fallback_pert

    # Optional h5ad blend for all perts
    if USE_ZCTRL_ALL and (gU in zctrl_all):
        z = (1.0 - float(ZCTRL_BETA)) * z + float(ZCTRL_BETA) * zctrl_all[gU]
        z = _norm1(z)

    # GenePT Z mapped into base-Z space (FIXED MAP)
    if USE_GENEPT and (GENEPT_WHERE in ["z", "both"]) and (genept_red_z is not None):
        v = genept_red_z.get(gU, None)
        z_ext = map_vec(v, W_geneptZ_to_Z)
        if z_ext is not None:
            z = (1.0 - float(GENEPT_GAMMA_Z)) * z + float(GENEPT_GAMMA_Z) * z_ext
            z = z / (np.linalg.norm(z) + 1e-12)

    # STRING seq/net Z mapped into base-Z space (FIXED MAP)
    if (STRING_WHERE in ["z", "both"]):
        if USE_STRING_SEQ and (string_seq_z is not None):
            v = string_seq_z.get(gU, None)
            z_ext = map_vec(v, W_sseqZ_to_Z)
            if z_ext is not None:
                z = (1.0 - float(STRING_GAMMA_Z)) * z + float(STRING_GAMMA_Z) * z_ext
                z = z / (np.linalg.norm(z) + 1e-12)

        if USE_STRING_NET and (string_net_z is not None):
            v = string_net_z.get(gU, None)
            z_ext = map_vec(v, W_snetZ_to_Z)
            if z_ext is not None:
                z = (1.0 - float(STRING_GAMMA_Z)) * z + float(STRING_GAMMA_Z) * z_ext
                z = z / (np.linalg.norm(z) + 1e-12)

    # DoRothEA Z
    if USE_DOROTHEA and (DORO_WHERE in ["z", "both"]) and (dorothea_z is not None):
        dp = dorothea_z.get(gU, None)
        if dp is not None:
            if DORO_FUSION_Z == "blend":
                z = _blend_z(z, dp.astype(np.float32), DORO_GAMMA_Z)
            else:
                z = np.concatenate([z, float(DORO_SCALE_Z) * dp.astype(np.float32)], axis=0)

    return z.astype(np.float32)

# -----------------------------
# Build U_out base: SVD + ctrl (blend recommended)
# -----------------------------
U_out_base = np.vstack([gene2emb_out_svd.get(str(g).upper(), emb_fallback_out) for g in gene_columns]).astype(np.float32)

if USE_UOUT_CTRL and (U_ctrl is not None):
    if UOUT_CTRL_MODE == "replace":
        U_out = U_ctrl.astype(np.float32)
        print("[exp] U_out = ctrl (replace)")

    elif UOUT_CTRL_MODE == "blend":
        a = float(UOUT_BLEND_ALPHA)

        # Align U_out_base into U_ctrl space before mixing
        print("[debug] pre-align cos(U_base, U_ctrl) =", mean_row_cos(U_out_base, U_ctrl))
        R = orth_procrustes(U_out_base, U_ctrl)
        U_base_aligned = (U_out_base @ R).astype(np.float32)
        print("[debug] post-align cos(U_base_aligned, U_ctrl) =", mean_row_cos(U_base_aligned, U_ctrl))

        U_out = (1.0 - a) * U_base_aligned + a * U_ctrl.astype(np.float32)
        U_out = _l2row(U_out)
        print(f"[exp] U_out = ALIGNED blend alpha={a:.3f} (SVD->ctrl)")

    elif UOUT_CTRL_MODE == "concat":
        U_out = np.concatenate([U_out_base, U_ctrl.astype(np.float32)], axis=1).astype(np.float32)
        print("[exp] U_out = concat(base, ctrl)")

    else:
        raise ValueError("UOUT_CTRL_MODE must be 'replace', 'concat', or 'blend'")
else:
    U_out = _l2row(U_out_base)
    print("[exp] U_out = base SVD")

# -----------------------------
# U blocks: CONCAT (safe), optionally skip GenePT-U if coverage low
# -----------------------------
# DoRothEA U
if USE_DOROTHEA and (DORO_WHERE in ["u", "both"]) and (dorothea_u is not None):
    U_doro = np.vstack([dorothea_u.get(str(g).upper(), np.zeros(int(DORO_DIM_U), np.float32)) for g in gene_columns])
    if DORO_FUSION_U == "concat":
        U_out = _concat_u(U_out, U_doro, DORO_SCALE_U)
        print("[exp] U_out = concat(+DoRothEA) ->", U_out.shape)

# STRING U
if STRING_WHERE in ["u", "both"]:
    if USE_STRING_SEQ and (string_seq_u is not None):
        U_sp = np.vstack([string_seq_u.get(str(g).upper(), np.zeros(int(STRING_DIM_U), np.float32)) for g in gene_columns])
        if STRING_FUSION_U == "concat":
            U_out = _concat_u(U_out, U_sp, STRING_SCALE_U)
            print("[exp] U_out = concat(+STRING seq) ->", U_out.shape)

    if USE_STRING_NET and (string_net_u is not None):
        U_sp = np.vstack([string_net_u.get(str(g).upper(), np.zeros(int(STRING_DIM_U), np.float32)) for g in gene_columns])
        if STRING_FUSION_U == "concat":
            U_out = _concat_u(U_out, U_sp, STRING_SCALE_U)
            print("[exp] U_out = concat(+STRING net) ->", U_out.shape)

# GenePT U
if USE_GENEPT and (GENEPT_WHERE in ["u", "both"]) and (genept_red_u is not None):
    have = sum([1 for g in gene_columns if str(g).upper() in genept_red_u])
    cov = have / max(1, len(gene_columns))
    print(f"[genept] U coverage = {cov:.3f}")
    if cov >= float(GENEPT_MIN_U_COVERAGE):
        U_gp = np.vstack([genept_red_u.get(str(g).upper(), np.zeros(int(GENEPT_DIM_U), np.float32)) for g in gene_columns])
        if GENEPT_FUSION_U == "concat":
            U_out = _concat_u(U_out, U_gp, GENEPT_SCALE_U)
            print("[exp] U_out = concat(+GenePT) ->", U_out.shape)
    else:
        print("[exp] skip GenePT-U (coverage too low)")

# =========================
# Align external embeddings into base spaces
# =========================
ALIGN_LAM = float(globals().get("ALIGN_LAM", 1e-2))

gene_columnsU = [str(g).upper() for g in gene_columns]
genes_neededU = [str(g).upper() for g in genes_needed]

# Base dicts for alignment targets
baseU_dict = {gene_columnsU[i]: U_out[i].copy() for i in range(len(gene_columnsU))}  # U_out is current base (80d)
baseZ_dict = {g: gene2emb_pert_svd[g].copy() for g in gene2emb_pert_svd.keys()}      # base Z space (80d) for output genes

# Fit alignment maps for sources that exist on gene_columns side
W_genept_to_U = None
W_genept_to_Z = None
W_sseq_to_U   = None
W_sseq_to_Z   = None
W_snet_to_U   = None
W_snet_to_Z   = None

if (genept_red_u is not None):
    W_genept_to_U = build_align_map_from_dicts(genept_red_u, baseU_dict, gene_columnsU, lam=ALIGN_LAM, name="genept->U")
    W_genept_to_Z = build_align_map_from_dicts(genept_red_u, baseZ_dict, gene_columnsU, lam=ALIGN_LAM, name="genept->Z")

if (string_seq_u is not None):
    W_sseq_to_U = build_align_map_from_dicts(string_seq_u, baseU_dict, gene_columnsU, lam=ALIGN_LAM, name="sseq->U")
    W_sseq_to_Z = build_align_map_from_dicts(string_seq_u, baseZ_dict, gene_columnsU, lam=ALIGN_LAM, name="sseq->Z")

if (string_net_u is not None):
    W_snet_to_U = build_align_map_from_dicts(string_net_u, baseU_dict, gene_columnsU, lam=ALIGN_LAM, name="snet->U")
    W_snet_to_Z = build_align_map_from_dicts(string_net_u, baseZ_dict, gene_columnsU, lam=ALIGN_LAM, name="snet->Z")

# =========================
# Align external embeddings into base spaces (FIXED)
# Fit Z-maps from *_z dicts, U-maps from *_u dicts
# =========================
ALIGN_LAM = float(globals().get("ALIGN_LAM", 1e-2))

gene_columnsU = [str(g).upper() for g in gene_columns]

# Base dict in Z-space (80d) for output genes
baseZ_dict = {g: gene2emb_pert_svd[g].copy() for g in gene2emb_pert_svd.keys()}  # keys are output genes

# Base dict in U-space uses CURRENT U_out (whatever dim it is right now)
baseU_dict = {gene_columnsU[i]: U_out[i].copy() for i in range(len(gene_columnsU))}

# --- Z maps (source = *_z dict) ---
W_geneptZ_to_Z = None
W_sseqZ_to_Z   = None
W_snetZ_to_Z   = None

if (genept_red_z is not None):
    W_geneptZ_to_Z = build_align_map_from_dicts(genept_red_z, baseZ_dict, gene_columnsU,
                                                lam=ALIGN_LAM, name="geneptZ->Z")

if (string_seq_z is not None):
    W_sseqZ_to_Z = build_align_map_from_dicts(string_seq_z, baseZ_dict, gene_columnsU,
                                              lam=ALIGN_LAM, name="sseqZ->Z")

if (string_net_z is not None):
    W_snetZ_to_Z = build_align_map_from_dicts(string_net_z, baseZ_dict, gene_columnsU,
                                              lam=ALIGN_LAM, name="snetZ->Z")

W_geneptU_to_U = None
W_sseqU_to_U   = None
W_snetU_to_U   = None

if (genept_red_u is not None):
    W_geneptU_to_U = build_align_map_from_dicts(genept_red_u, baseU_dict, gene_columnsU,
                                                lam=ALIGN_LAM, name="geneptU->U")

if (string_seq_u is not None):
    W_sseqU_to_U = build_align_map_from_dicts(string_seq_u, baseU_dict, gene_columnsU,
                                              lam=ALIGN_LAM, name="sseqU->U")

if (string_net_u is not None):
    W_snetU_to_U = build_align_map_from_dicts(string_net_u, baseU_dict, gene_columnsU,
                                              lam=ALIGN_LAM, name="snetU->U")

def map_vec(v, W):
    if (W is None) or (v is None):
        return None
    v = v.astype(np.float32)
    if v.shape[0] != W.shape[0]:
        return None
    return (v @ W).astype(np.float32)
# -----------------------------
# Final Z_train
# -----------------------------
Z_train = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)

SVD k: 80 gene_emb_all: (5127, 80)
Effective dims: d_pert= 80 d_out= 80
train perts not in gene_columns: 8. Example: ['BRD4', 'CHD4', 'DNAJA3', 'INO80', 'KAT8', 'KDM4A', 'PMEL', 'SETD1A']
val perts not in gene_columns: 8. Example: ['SMARCB1', 'PSMA1', 'CUL1', 'FLT4', 'FOXH1', 'HK2', 'TRAM2', 'DPH2']
[h5ad] cache built: n_cells= 17882 n_genes= 19226 n_ctrl= 1026
[h5ad] embedded missing perts: 16 of 16
[h5ad] embedded ALL perts for blending: 140 of 140
[h5ad] U_ctrl: (5127, 80)
[genept] reduced: z_avail= 138 u_avail= 4998 dim_z= 80 dim_u= 64
[dorothea] built UniProt->symbol map from STRING: 474839
[dorothea] parsed cols: ['source', 'target', 'is_directed', 'is_stimulation', 'is_inhibition', 'consensus_direction', 'consensus_stimulation', 'consensus_inhibition']
[dorothea] edges loaded: 15267 kept(in-gene-set): 4876
[dorothea] detected UniProt-like IDs: True | mapping size: 474839
[dorothea] Z policy (train perts): row=5 col=28 fb=47 (n=80)
[dorothea] Z policy (genes_needed): row=13 col=4

In [7]:
# =========================
# Add external info to U_out in the SAME 80d space (mapped)
# =========================
def add_mapped_U(U_out, ext_dict, W_to_U, scale, name):
    if (ext_dict is None) or (W_to_U is None):
        return U_out
    U_add = np.zeros_like(U_out, dtype=np.float32)
    for i, g in enumerate(gene_columnsU):
        v = ext_dict.get(g, None)
        if v is not None:
            U_add[i] = map_vec(v, W_to_U)
    U_out2 = U_out + float(scale) * U_add
    U_out2 = _l2row(U_out2)
    print(f"[exp] U_out += mapped {name} scale={float(scale):.3f}")
    return U_out2

U_out = add_mapped_U(U_out, genept_red_u,  W_genept_to_U, GENEPT_SCALE_U, "GenePT")
U_out = add_mapped_U(U_out, string_seq_u,  W_sseq_to_U,   STRING_SCALE_U, "STRING seq")
U_out = add_mapped_U(U_out, string_net_u,  W_snet_to_U,   STRING_SCALE_U, "STRING net")

[exp] U_out += mapped GenePT scale=0.050
[exp] U_out += mapped STRING seq scale=0.100
[exp] U_out += mapped STRING net scale=0.100


SVD k: 80 gene_emb_all: (5127, 80)
Effective dims: d_pert= 80 d_out= 80
train perts not in gene_columns: 8. Example: ['BRD4', 'CHD4', 'DNAJA3', 'INO80', 'KAT8', 'KDM4A', 'PMEL', 'SETD1A']
val perts not in gene_columns: 8. Example: ['SMARCB1', 'PSMA1', 'CUL1', 'FLT4', 'FOXH1', 'HK2', 'TRAM2', 'DPH2']
[h5ad] cache built: n_cells= 17882 n_genes= 19226 n_ctrl= 1026
[h5ad] embedded missing perts: 16 of 16
[h5ad] embedded ALL perts for blending: 140 of 140
[h5ad] U_ctrl: (5127, 80)
[genept] reduced: z_avail= 138 u_avail= 4998 dim_z= 64 dim_u= 64
[dorothea] built UniProt->symbol map from STRING: 474839
[dorothea] parsed cols: ['source', 'target', 'is_directed', 'is_stimulation', 'is_inhibition', 'consensus_direction', 'consensus_stimulation', 'consensus_inhibition']
[dorothea] edges loaded: 15267 kept(in-gene-set): 4876
[dorothea] detected UniProt-like IDs: True | mapping size: 474839
[dorothea] Z policy (train perts): row=5 col=28 fb=47 (n=80)
[dorothea] Z policy (genes_needed): row=13 col=42 fb=85 (n=140)
[dorothea] degree stats: outdeg>0: 125 indeg>0: 1746 G: 5127
[dorothea] directed_svd built: dim_z= 64 dim_u= 64 split_pos_neg= True unsigned_ch= False signed_only= False
[dorothea] tables: z_avail= 140 u_avail= 5127 dim_z= 64 dim_u= 64
[string] reduced: seq_z= 138 seq_u= 4986 net_z= 138 net_u= 4986 dim_z= 64 dim_u= 64
[impute Z] genept_red_z: before=138/140 filled=2 after=140/140
[impute Z] string_seq_z: before=138/140 filled=2 after=140/140
[impute Z] string_net_z: before=138/140 filled=2 after=140/140
[impute U] genept_red_u: before=4998/5127 filled=129 after=5127/5127
[impute U] string_seq_u: before=4986/5127 filled=141 after=5127/5127
[impute U] string_net_u: before=4986/5127 filled=141 after=5127/5127
[exp] U_out = ctrl (replace)
[exp] U_out = concat(+GenePT) -> (5127, 144)
[exp] U_out = concat(+DoRothEA) -> (5127, 208)
[exp] U_out = concat(+STRING seq) -> (5127, 272)
[exp] U_out = concat(+STRING net) -> (5127, 336)
EXP_NAME: baseline_fixed_missing
U_out: (5127, 336) Z_train: (80, 336)

In [8]:
# =========================
# POST-COMPRESS U_out (denoise external concat)
# =========================
POST_REDUCE_UOUT = True
UOUT_POST_DIM = 80   # try 80 first (baseline-compatible); also try 128 later

if POST_REDUCE_UOUT:
    # U_out is (G, D_total). Reduce to (G, UOUT_POST_DIM)
    U_tmp = U_out.astype(np.float32)

    # center=False behavior of TruncatedSVD is fine here; we just want a low-rank basis
    svd_u = TruncatedSVD(n_components=int(UOUT_POST_DIM), random_state=SEED)
    U_red = svd_u.fit_transform(U_tmp).astype(np.float32)  # (G, k)

    # row-normalize to keep scale sane
    U_red = U_red / (np.linalg.norm(U_red, axis=1, keepdims=True) + 1e-12)

    U_out = U_red
    print(f"[post] U_out post-reduced to {U_out.shape} (explained_var_sum={svd_u.explained_variance_ratio_.sum():.4f})")

print("EXP_NAME:", EXP_NAME)
print("U_out:", U_out.shape, "Z_train:", Z_train.shape)

[post] U_out post-reduced to (5127, 80) (explained_var_sum=0.9730)
EXP_NAME: baseline_fixed_missing
U_out: (5127, 80) Z_train: (80, 80)


In [ ]:
def gate_smoothstep(x, a = GATE_A, b = GATE_B):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def per_row_weighted_l1_like(delta_true: torch.Tensor, delta_pred: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    """
    Same logic as weighted_l1_like, but returns (N,) per-row.
    """
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)  # (N, G)
    err = torch.abs(delta_pred - delta_true)                        # (N, G)
    num = torch.sum(w * err, dim=1)                                 # (N,)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)                 # (N,)
    return num / den                                                # (N,)

import torch

def weighted_l1_like_rowweighted(
    delta_true: torch.Tensor,     # (N, G)
    delta_pred: torch.Tensor,     # (N, G)
    baseline_wmae: torch.Tensor,  # (N,)
    *,
    eps: float = 1e-8,
    mode: str = "inv_sqrt",       # "inv", "inv_sqrt", "inv_log"
    clamp_min: float = 0.5,
    clamp_max: float = 3.0,
) -> torch.Tensor:
    """
    Weight idea:
      - baseline_wmae small => ratio metric is unforgiving => upweight that row
      - baseline_wmae large => easier => downweight a bit

    Returns: scalar loss
    """
    per_row = per_row_weighted_l1_like(delta_true, delta_pred, eps=eps)  # (N,)

    b = baseline_wmae.to(delta_true.device).to(delta_true.dtype)

    if mode == "inv":
        w = 1.0 / (b + eps)
    elif mode == "inv_sqrt":
        w = 1.0 / torch.sqrt(b + eps)
    elif mode == "inv_log":
        w = 1.0 / torch.log1p(b + eps)
    else:
        raise ValueError(f"Unknown mode={mode}")

    # clamp to prevent a few rows from dominating training
    w = torch.clamp(w, min=clamp_min, max=clamp_max)

    # normalized weighted mean (stable)
    return torch.sum(w * per_row) / torch.clamp(torch.sum(w), min=eps)

def weighted_cosine_per_row_torch(dt: torch.Tensor, dp: torch.Tensor, w: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    # dt, dp, w: (B, G)
    wa = w * dt
    wb = w * dp
    num = torch.sum(wa * wb, dim=1)
    da  = torch.sqrt(torch.sum(wa * wa, dim=1))
    db  = torch.sqrt(torch.sum(wb * wb, dim=1))
    denom = torch.clamp(da * db, min=eps)
    return num / denom  # (B,)


In [10]:
class BilinearDeltaModel(nn.Module):
    def __init__(self, d_pert, d_out, rank_r, dropout):
        super().__init__()
        self.rank_r = rank_r

        self.proj_p = nn.Sequential(
            nn.Linear(d_pert, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.proj_o = nn.Sequential(
            nn.Linear(d_out, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # biases
        self.bias_global = nn.Parameter(torch.zeros(1))
        self.bias_gene = None  # set via set_gene_bias

    def set_gene_bias(self, G):
        dev = next(self.parameters()).device
        self.bias_gene = torch.nn.Parameter(torch.zeros(G, device=dev))
        self.bias_global = torch.nn.Parameter(torch.zeros(1, device=dev))

    def forward(self, z_pert, u_out):
        p = self.proj_p(z_pert)    # (B, R)
        o = self.proj_o(u_out)     # (G, R)
        y = p @ o.T                # (B, G)
        y = y + self.bias_gene[None, :] + self.bias_global
        return y

In [11]:
Y = D_train.astype(np.float32)
G = Y.shape[1]
N = Y.shape[0]

Uo_t = torch.tensor(U_out, device=device) # (G, d_out)
Zt = torch.tensor(Z_train, device=device) # (N, d_pert)
Yt = torch.tensor(Y, device=device) # (N, G)

print("N:", N, "G:", G, "device:", device)

N: 80 G: 5127 device: cuda


In [12]:
gt_df = pd.read_csv("Data/training_data_ground_truth_table.csv")
sol_aligned = gt_df.set_index("pert_id").loc[train_genes].reset_index()

# Keep only needed columns for speed: pert_id + genes + weights + baseline
w_cols = [f"w_{g}" for g in gene_columns]
sol_aligned = sol_aligned[["pert_id"] + list(gene_columns) + w_cols + ["baseline_wmae"]]
baseline_wmae = sol_aligned["baseline_wmae"].to_numpy(np.float32)
baseline_wmae_t = torch.tensor(baseline_wmae, device=device, dtype=torch.float32)

W_gene = sol_aligned[w_cols].to_numpy(np.float32)   # (N, G)

W64 = W_gene.astype(np.float64)
row_sums = W64.sum(axis=1, keepdims=True)
W64 *= (G / np.maximum(row_sums, 1e-300))
W64[:, -1] += (G - W64.sum(axis=1))
W_gene = W64.astype(np.float32)

In [13]:
from myllia_metric import score as kaggle_score

def score_delta(dt, dp, idx):
    dt = np.asarray(dt, np.float64)
    dp = np.asarray(dp, np.float64)
    w  = W_gene[idx].astype(np.float64, copy=False)
    base = baseline_wmae[idx].astype(np.float64, copy=False)

    abs_err = np.abs(dt - dp)
    pred_wmae = np.mean(abs_err * w, axis=1)
    pred_wmae = np.maximum(pred_wmae, 1e-12)
    base = np.maximum(base, 1e-12)

    terms = np.log2(base / pred_wmae)
    terms = np.minimum(terms, 5.0)
    sum_wmae = float(np.sum(terms))
    mean_term = float(np.mean(terms))

    a = dp.ravel()
    b = dt.ravel()
    x = np.maximum(np.abs(a), np.abs(b))
    t = np.clip(x / 0.2, 0.0, 1.0)
    w_gate = t * t * (3.0 - 2.0 * t)
    w2 = w_gate * w_gate

    num = np.sum(w2 * a * b)
    den = np.sqrt(np.sum(w2 * a * a)) * np.sqrt(np.sum(w2 * b * b))
    wcos = 0.0 if den < 1e-12 else float(num / den)
    wcos_pos = max(0.0, wcos)

    raw = sum_wmae * wcos_pos
    score = round(raw, 5)

    # normalized (scale-free) metric
    score_per_row = mean_term * wcos_pos

    return {
        "score": score,
        "raw": float(raw),
        "sum_wmae": sum_wmae,
        "mean_term": mean_term,
        "wcos": wcos,
        "score_per_row": float(score_per_row),
        "n_rows": int(len(idx)),
    }

In [14]:
# =============================
# EXTRA H5AD DATA: BOOTSTRAP DELTAS (precompute once)
# =============================
boot_raw = None
h5_mean_raw = None
boot_ok = None

if AUGMENT_H5AD:
    cache = load_h5ad_ctrl_cache(H5AD_PATH, gene_columns)

    Xn = cache["Xn"]               # sparse (n_cells, 19226) normalized CPM10K + log2
    out_idx = cache["out_idx"]     # (5127,)
    obs_pertU = cache["obs_pertU"] # (n_cells,) uppercase pert labels
    ctrl_mask = cache["ctrl_mask"]

    # control mean over output genes (in same h5ad normalized space)
    ctrl_mean = Xn[ctrl_mask][:, out_idx].mean(axis=0)
    if sparse.issparse(ctrl_mean):
        ctrl_mean = ctrl_mean.A
    ctrl_mean = np.asarray(ctrl_mean).ravel().astype(np.float32)  # (5127,)

    N = len(train_genes)
    G = len(gene_columns)

    h5_mean_raw = np.zeros((N, G), dtype=np.float32)
    boot_raw = np.zeros((N, BOOT_K, G), dtype=np.float32)
    boot_ok = np.zeros((N,), dtype=np.int32)

    rng = np.random.RandomState(BOOT_SEED)

    for i, g in enumerate(train_genes.tolist()):
        gU = str(g).upper()
        rows = np.where(obs_pertU == gU)[0]
        if len(rows) == 0:
            continue

        boot_ok[i] = 1

        # mean delta using ALL cells for this pert
        Xi = Xn[rows][:, out_idx].mean(axis=0)
        if sparse.issparse(Xi):
            Xi = Xi.A
        Xi = np.asarray(Xi).ravel().astype(np.float32)
        h5_mean_raw[i] = (Xi - ctrl_mean)

        # bootstraps
        for k in range(BOOT_K):
            samp = rng.choice(rows, size=min(int(BOOT_M), len(rows)), replace=True)
            Xk = Xn[samp][:, out_idx].mean(axis=0)
            if sparse.issparse(Xk):
                Xk = Xk.A
            Xk = np.asarray(Xk).ravel().astype(np.float32)
            boot_raw[i, k] = (Xk - ctrl_mean)

    print("[aug] boot_ok:", int(boot_ok.sum()), "/", N)
    print("[aug] h5_mean_raw:", h5_mean_raw.shape, "boot_raw:", boot_raw.shape)

[aug] boot_ok: 80 / 80
[aug] h5_mean_raw: (80, 5127) boot_raw: (80, 32, 5127)


In [ ]:
def apply_shrink(pred, baseline, alpha):
    # pred: (B,G) ; baseline: (G,)
    return float(alpha) * pred + (1.0 - float(alpha)) * baseline[None, :]

def train_one_fold(tr_idx, va_idx, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT
    ).to(device)

    model.set_gene_bias(G)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    aug_enabled = (AUGMENT_H5AD and (boot_raw is not None) and (h5_mean_raw is not None) and (boot_ok is not None))

    slopes_t = None
    intercepts_t = None
    rng_aug = None

    if aug_enabled:
        ok_mask = (boot_ok[tr_idx] == 1)
        tr_fit = tr_idx[ok_mask]

        if len(tr_fit) >= AUG_MIN_FIT_PERTS:
            A = h5_mean_raw[tr_fit].astype(np.float32)  # (n_fit, G) in h5ad-space
            B = Y[tr_fit].astype(np.float32)            # (n_fit, G) in means-space

            # per-gene affine: B ~= slope * A + intercept
            Am = A.mean(axis=0)
            Bm = B.mean(axis=0)
            Av = ((A - Am[None, :]) ** 2).mean(axis=0) + 1e-6
            Cov = ((A - Am[None, :]) * (B - Bm[None, :])).mean(axis=0)

            slopes = (Cov / Av).astype(np.float32)
            slopes = np.clip(slopes, float(AUG_SLOPE_CLAMP_MIN), float(AUG_SLOPE_CLAMP_MAX)).astype(np.float32)
            intercepts = (Bm - slopes * Am).astype(np.float32)

            slopes_t = torch.tensor(slopes[None, :], device=device, dtype=torch.float32)       # (1, G)
            intercepts_t = torch.tensor(intercepts[None, :], device=device, dtype=torch.float32) # (1, G)

            rng_aug = np.random.RandomState(seed + 2027)
            #print("[aug] enabled: fit_perts=", len(tr_fit), "slope_mean=", float(slopes.mean()), "slope_std=", float(slopes.std()))
        else:
            aug_enabled = False
            print("[aug] disabled: not enough fit perts in this fold:", len(tr_fit))

    best_score = -1e18
    best_alpha = 0.0
    best_epoch = 0
    best_state = None
    best_va_pred = None  # unshrunk predictions at best checkpoint
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)

            if aug_enabled and (slopes_t is not None):
                kk = rng_aug.randint(0, BOOT_K, size=len(b))
                dt_raw_np = boot_raw[b, kk, :]  # (B, G) in h5ad-space
                dt_raw_t = torch.tensor(dt_raw_np, device=device, dtype=torch.float32)

                # map to means-space using train-only affine
                dt_aug = dt_raw_t * slopes_t + intercepts_t  # (B, G)

                # mix with original means target for stability
                dt_b = (1.0 - float(AUG_P)) * Yt.index_select(0, b_t) + float(AUG_P) * dt_aug
            else:
                dt_b = Yt.index_select(0, b_t)               # (B, G)

            bw_b = baseline_wmae_t.index_select(0, b_t)     # (B,)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t), Uo_t).detach().cpu().numpy().astype(np.float32)

            va_true = Y[va_idx]

            # alpha sweep (0..0.7) on this fold
            sc_best = -1e18
            a_best = 0.0
            for a in ALPHA_GRID:
                pred_a = apply_shrink(va_pred, delta_baseline, float(a))
                sc = score_delta(va_true, pred_a, va_idx)["score"]
                if sc > sc_best:
                    sc_best = sc
                    a_best = float(a)

            if sc_best > best_score:
                best_score = sc_best
                best_alpha = a_best
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_va_pred = va_pred.copy()
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_alpha, best_epoch, best_state, best_va_pred

# --- CV: collect OOF preds and optimize a GLOBAL alpha on OOF only ---
kf = KFold(n_splits=8, shuffle=True, random_state=SEED)

oof_pred = np.zeros_like(Y, dtype=np.float32)
oof_hit  = np.zeros((N,), dtype=np.int32)

fold_scores = []          # mean score per fold across seeds
fold_scores_seeds = []
fold_epochs = []          # median/mean epoch across seeds per fold (pick one)
fold_alphas = []          # mean alpha across seeds per fold (just for logging)

for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):

    # Collect per-seed results for this fold
    seed_scores = []
    seed_alphas = []
    seed_epochs = []
    seed_va_preds = []

    for s in MODEL_SEEDS:
        best_score, best_alpha, best_epoch, best_state, best_va_pred = train_one_fold(tr_idx, va_idx, seed=int(s))

        seed_scores.append(float(best_score))
        seed_alphas.append(float(best_alpha))
        seed_epochs.append(int(best_epoch))
        seed_va_preds.append(best_va_pred.astype(np.float32, copy=False))

    fold_score_mean = float(np.mean(seed_scores))
    fold_scores_seeds.append(seed_scores)

    # Logging-only summaries
    fold_alphas.append(float(np.mean(seed_alphas)))
    fold_epochs.append(int(np.median(seed_epochs)))

    va_pred_mean = np.mean(np.stack(seed_va_preds, axis=0), axis=0).astype(np.float32)  # (B, G)

    oof_pred[va_idx] = va_pred_mean
    oof_hit[va_idx] += 1
    fold_factor = float(N / len(va_idx))  # ~8.0 for 8-fold CV
    seeds_str = ", ".join([f"{sc * fold_factor:.6f}" for sc in seed_scores])
    fold_score_scaled = fold_score_mean * fold_factor
    fold_scores.append(fold_score_scaled)
    print(
        f"fold {fold}: score_mean={fold_score_scaled:.6f} "
        f"scores=[{seeds_str}] "
        f"alpha_mean={np.mean(seed_alphas):.3f} "
        f"epoch_median={int(np.median(seed_epochs))}"
    )

if not np.all(oof_hit == 1):
    print("[warn] OOF coverage not 1 everywhere. min/max:", int(oof_hit.min()), int(oof_hit.max()))

print("cv mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))

EPOCHS_MED = int(np.median(fold_epochs))
print("median best_epoch =", EPOCHS_MED)

best_global_alpha = 0.0
best_global_score = -1e18

all_idx = np.arange(N, dtype=np.int64)

for a in ALPHA_GRID:
    pred_a = apply_shrink(oof_pred, delta_baseline, float(a))
    sc = score_delta(Y, pred_a, all_idx)["score"]
    if sc > best_global_score:
        best_global_score = sc
        best_global_alpha = float(a)

print("OOF global alpha:", best_global_alpha, "OOF score:", best_global_score)
ALPHA_SHRINK = best_global_alpha

fold 1: score_mean=6.057813 scores=[5.911680, 5.565760, 6.696000] alpha_mean=0.860 epoch_median=60
fold 2: score_mean=4.459067 scores=[4.678000, 4.353920, 4.345280] alpha_mean=0.860 epoch_median=40
fold 3: score_mean=4.614907 scores=[4.377920, 4.771520, 4.695280] alpha_mean=0.860 epoch_median=45
fold 4: score_mean=3.842347 scores=[3.836080, 3.733920, 3.957040] alpha_mean=0.860 epoch_median=35
fold 5: score_mean=4.606320 scores=[4.591760, 4.457280, 4.769920] alpha_mean=0.860 epoch_median=45
fold 6: score_mean=6.719520 scores=[7.337680, 6.364720, 6.456160] alpha_mean=0.860 epoch_median=45
fold 7: score_mean=5.943760 scores=[5.756800, 6.000240, 6.074240] alpha_mean=0.860 epoch_median=25
fold 8: score_mean=3.583387 scores=[3.586160, 3.699760, 3.464240] alpha_mean=0.833 epoch_median=30
cv mean: 4.978389999999999 std: 1.0550775649580355
median best_epoch = 42
OOF global alpha: 0.86 OOF score: 4.92531


|Type|cv mean|OOF score|
|-------|-----------|-------------|
|Dor Gen All|5.74874|5.68997|
|No Dorothea |5.56606| 5.42451 |
|No Genept|5.536723|5.3467|
| Just Dorothea|5.44974|5.23232|
|Baseline| 5.417313| 5.18041|

fold 1: score_mean=6.148160 scores=[6.037360, 6.146800, 6.260320] alpha_mean=0.860 epoch_median=40
fold 2: score_mean=5.349253 scores=[4.925520, 5.686880, 5.435360] alpha_mean=0.860 epoch_median=145
fold 3: score_mean=4.252480 scores=[4.229520, 4.161760, 4.366160] alpha_mean=0.750 epoch_median=65
fold 4: score_mean=4.946373 scores=[4.911840, 5.031840, 4.895440] alpha_mean=0.860 epoch_median=135
fold 5: score_mean=5.389707 scores=[5.553840, 5.733360, 4.881920] alpha_mean=0.860 epoch_median=190
fold 6: score_mean=7.761893 scores=[7.797200, 7.578720, 7.909760] alpha_mean=0.847 epoch_median=80
fold 7: score_mean=7.413013 scores=[7.478400, 7.368480, 7.392160] alpha_mean=0.847 epoch_median=30
fold 8: score_mean=4.729040 scores=[4.666080, 4.761520, 4.759520] alpha_mean=0.860 epoch_median=40
cv mean: 5.74874 std: 1.1828460003642813
median best_epoch = 72
OOF global alpha: 0.86 OOF score: 5.68997

In [ ]:
def fit_full_model(seed, epochs_fixed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT
    ).to(device)

    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    aug_enabled = False
    slopes_t = None
    intercepts_t = None
    rng_aug = None

    if ("AUGMENT_H5AD" in globals()) and AUGMENT_H5AD:
        assert (boot_raw is not None) and (h5_mean_raw is not None) and (boot_ok is not None), \
            "AUGMENT_H5AD=True but boot_raw/h5_mean_raw/boot_ok not built. Run the bootstrap precompute cell."

        fit_idx = np.where(boot_ok == 1)[0]
        if len(fit_idx) >= AUG_MIN_FIT_PERTS:
            A = h5_mean_raw[fit_idx].astype(np.float32)  # (n_fit, G) h5ad-space
            B = Y[fit_idx].astype(np.float32)            # (n_fit, G) means-space

            Am = A.mean(axis=0)
            Bm = B.mean(axis=0)
            Av = ((A - Am[None, :]) ** 2).mean(axis=0) + 1e-6
            Cov = ((A - Am[None, :]) * (B - Bm[None, :])).mean(axis=0)

            slopes = (Cov / Av).astype(np.float32)
            slopes = np.clip(slopes, float(AUG_SLOPE_CLAMP_MIN), float(AUG_SLOPE_CLAMP_MAX)).astype(np.float32)
            intercepts = (Bm - slopes * Am).astype(np.float32)

            slopes_t = torch.tensor(slopes[None, :], device=device, dtype=torch.float32)          # (1, G)
            intercepts_t = torch.tensor(intercepts[None, :], device=device, dtype=torch.float32)  # (1, G)

            rng_aug = np.random.RandomState(seed + 2027)
            aug_enabled = True
            print(f"[aug/full] enabled: fit_perts={len(fit_idx)} slope_mean={float(slopes.mean()):.6f} slope_std={float(slopes.std()):.6f}")
        else:
            print(f"[aug/full] disabled: not enough fit perts with h5ad coverage ({len(fit_idx)})")

    all_idx = np.arange(N, dtype=np.int64)

    for epoch in range(1, int(epochs_fixed) + 1):
        model.train()
        perm = np.arange(N)
        np.random.shuffle(perm)

        for start in range(0, N, BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)

            if aug_enabled and (slopes_t is not None):
                kk = rng_aug.randint(0, BOOT_K, size=len(b))
                dt_raw_np = boot_raw[b, kk, :]  # (B, G) h5ad-space
                dt_raw_t = torch.tensor(dt_raw_np, device=device, dtype=torch.float32)

                dt_aug = dt_raw_t * slopes_t + intercepts_t  # (B, G) means-space
                dt_b = (1.0 - float(AUG_P)) * Yt.index_select(0, b_t) + float(AUG_P) * dt_aug
            else:
                dt_b = Yt.index_select(0, b_t)

            bw_b = baseline_wmae_t.index_select(0, b_t)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        # -----------------------------
        # Monitor score in submission space (means-space + shrink)
        # -----------------------------
        if (epoch % EVAL_EVERY == 0) or (epoch == int(epochs_fixed)):
            model.eval()
            with torch.no_grad():
                pred_np = model(Zt, Uo_t).detach().cpu().numpy().astype(np.float32)

            pred_np = apply_shrink(pred_np, delta_baseline, float(ALPHA_SHRINK))

            # IMPORTANT: score_delta_fast expects idx
            s = score_delta(Y, pred_np, all_idx)

            if "wcos" in s:
                print(f"[seed {seed}] epoch={epoch:4d} train_score={s['score']:.6f} wcos={s['wcos']:.6f} mean_term={s.get('mean_term', float('nan')):.6f} alpha={float(ALPHA_SHRINK):.3f}")
            else:
                print(f"[seed {seed}] epoch={epoch:4d} train_score={s['score']:.6f} alpha={float(ALPHA_SHRINK):.3f}")

    model.eval()
    return model


# Refit ensemble on ALL perts, using CV-calibrated epoch + OOF-calibrated alpha
models = [fit_full_model(int(sd), epochs_fixed=int(EPOCHS_MED)) for sd in MODEL_SEEDS]
print("Refit models:", len(models))


def predict_delta_gene(gene_symbol: str) -> np.ndarray:
    z = torch.tensor(emb_pert(gene_symbol)[None, :].astype(np.float32), device=device)
    preds = []
    with torch.no_grad():
        for m in models:
            y = m(z, Uo_t).detach().cpu().numpy().astype(np.float32)[0]
            preds.append(y)
    yhat = np.mean(np.stack(preds, axis=0), axis=0).astype(np.float32)
    yhat = apply_shrink(yhat[None, :], delta_baseline, float(ALPHA_SHRINK))[0].astype(np.float32)
    return yhat


# Build submission from sample_submission.csv
sub = df_sub.copy()
sub["pert_id"] = sub["pert_id"].astype(str)
sub_gene_cols = [c for c in sub.columns if c != "pert_id"]

# ensure order matches gene_columns
idx_map = {str(g).upper(): i for i, g in enumerate(gene_columns)}
perm = [idx_map[str(g).upper()] for g in sub_gene_cols]

# fill default baseline for unknown test perts
sub.loc[:, sub_gene_cols] = np.tile(delta_baseline[perm][None, :], (len(sub), 1))

# fill known val perts (pert_1..pert_60)
hit = 0
for pid, gene in val_map.items():
    vec = predict_delta_gene(gene)[perm]
    m = (sub["pert_id"] == str(pid))
    if m.any():
        sub.loc[m, sub_gene_cols] = vec[None, :]
        hit += int(m.sum())

#out_path = "Submission_Dorothea.csv"
#sub.to_csv(out_path, index=False)
#print("[ok] wrote:", out_path, "| filled:", hit)

[aug/full] enabled: fit_perts=80 slope_mean=0.980505 slope_std=0.062581
[seed 90] epoch=   5 train_score=0.204520 wcos=0.370923 mean_term=0.006892 alpha=0.860
[seed 90] epoch=  10 train_score=4.987390 wcos=0.385185 mean_term=0.161850 alpha=0.860
[seed 90] epoch=  15 train_score=7.054840 wcos=0.370883 mean_term=0.237772 alpha=0.860
[seed 90] epoch=  20 train_score=7.623750 wcos=0.360050 mean_term=0.264677 alpha=0.860
[seed 90] epoch=  25 train_score=7.950260 wcos=0.360303 mean_term=0.275819 alpha=0.860
[seed 90] epoch=  30 train_score=8.164080 wcos=0.361341 mean_term=0.282423 alpha=0.860
[seed 90] epoch=  35 train_score=8.387150 wcos=0.365140 mean_term=0.287121 alpha=0.860
[seed 90] epoch=  40 train_score=8.676480 wcos=0.367604 mean_term=0.295035 alpha=0.860
[seed 90] epoch=  45 train_score=8.949110 wcos=0.372454 mean_term=0.300343 alpha=0.860
[seed 90] epoch=  50 train_score=9.374640 wcos=0.376723 mean_term=0.311058 alpha=0.860
[seed 90] epoch=  55 train_score=9.865080 wcos=0.386110 mean_term=0.319374 alpha=0.860
[seed 90] epoch=  60 train_score=10.450860 wcos=0.394671 mean_term=0.330999 alpha=0.860
[seed 90] epoch=  65 train_score=11.120210 wcos=0.403445 mean_term=0.344539 alpha=0.860
[seed 90] epoch=  70 train_score=11.722880 wcos=0.413204 mean_term=0.354633 alpha=0.860
[seed 90] epoch=  72 train_score=11.993340 wcos=0.417029 mean_term=0.359487 alpha=0.860
[aug/full] enabled: fit_perts=80 slope_mean=0.980505 slope_std=0.062581
[seed 70] epoch=   5 train_score=0.315610 wcos=0.378215 mean_term=0.010431 alpha=0.860
[seed 70] epoch=  10 train_score=4.983450 wcos=0.385221 mean_term=0.161707 alpha=0.860
[seed 70] epoch=  15 train_score=6.860000 wcos=0.367138 mean_term=0.233563 alpha=0.860
[seed 70] epoch=  20 train_score=7.518720 wcos=0.357496 mean_term=0.262895 alpha=0.860
[seed 70] epoch=  25 train_score=7.845570 wcos=0.357468 mean_term=0.274345 alpha=0.860
[seed 70] epoch=  30 train_score=8.020930 wcos=0.358138 mean_term=0.279953 alpha=0.860
[seed 70] epoch=  35 train_score=8.217690 wcos=0.361223 mean_term=0.284370 alpha=0.860
[seed 70] epoch=  40 train_score=8.494360 wcos=0.364674 mean_term=0.291163 alpha=0.860
[seed 70] epoch=  45 train_score=8.793090 wcos=0.368566 mean_term=0.298219 alpha=0.860
[seed 70] epoch=  50 train_score=9.097810 wcos=0.374084 mean_term=0.304003 alpha=0.860
[seed 70] epoch=  55 train_score=9.417760 wcos=0.379653 mean_term=0.310078 alpha=0.860
[seed 70] epoch=  60 train_score=9.868820 wcos=0.385945 mean_term=0.319632 alpha=0.860
[seed 70] epoch=  65 train_score=10.337160 wcos=0.394603 mean_term=0.327454 alpha=0.860
[seed 70] epoch=  70 train_score=10.927830 wcos=0.403170 mean_term=0.338810 alpha=0.860
[seed 70] epoch=  72 train_score=11.169930 wcos=0.406811 mean_term=0.343216 alpha=0.860
[aug/full] enabled: fit_perts=80 slope_mean=0.980505 slope_std=0.062581
[seed 80] epoch=   5 train_score=0.333800 wcos=0.388792 mean_term=0.010732 alpha=0.860
[seed 80] epoch=  10 train_score=5.117450 wcos=0.387504 mean_term=0.165077 alpha=0.860
[seed 80] epoch=  15 train_score=6.950820 wcos=0.366965 mean_term=0.236767 alpha=0.860
[seed 80] epoch=  20 train_score=7.594270 wcos=0.358888 mean_term=0.264507 alpha=0.860
[seed 80] epoch=  25 train_score=7.864560 wcos=0.357690 mean_term=0.274838 alpha=0.860
[seed 80] epoch=  30 train_score=8.022200 wcos=0.358264 mean_term=0.279898 alpha=0.860
[seed 80] epoch=  35 train_score=8.212910 wcos=0.360182 mean_term=0.285026 alpha=0.860
[seed 80] epoch=  40 train_score=8.441230 wcos=0.364861 mean_term=0.289193 alpha=0.860
[seed 80] epoch=  45 train_score=8.755740 wcos=0.368554 mean_term=0.296962 alpha=0.860
[seed 80] epoch=  50 train_score=9.038620 wcos=0.375385 mean_term=0.300979 alpha=0.860
[seed 80] epoch=  55 train_score=9.499490 wcos=0.380927 mean_term=0.311722 alpha=0.860
[seed 80] epoch=  60 train_score=9.955270 wcos=0.387267 mean_term=0.321331 alpha=0.860
[seed 80] epoch=  65 train_score=10.488660 wcos=0.397158 mean_term=0.330116 alpha=0.860
[seed 80] epoch=  70 train_score=11.050110 wcos=0.403302 mean_term=0.342489 alpha=0.860
[seed 80] epoch=  72 train_score=11.316990 wcos=0.408244 mean_term=0.346514 alpha=0.860
Refit models: 3